In [ ]:
import pandas as pd, numpy as np

%load_ext autoreload
%autoreload 2

import hces_utils
from hces_utils import weighted_avg_and_std, weighted_summary

!date

# HCES Data Extraction
We have HCES data to inform rice consumption (of PSD-distributed and non-PSD rice) by women and birthing people of 
reproductive age (WBPRA) and U5 children in India. We need to do further investigation as to what percentage of PSD-
distributed rice has been fortified with iron and folate.

Old HCES data and documentation is saved on Sharepoint here: https://uwnetid.sharepoint.com/:f:/r/sites/ihme_simulation_science_team/Shared%20Documents/Research/LSFF/07_Data/HCES_22_data_files?csf=1&web=1&e=wmxQSD; new docs on HCES 2023-24 are stored in
`/snfs1/Project/simulation_science/IND_HOUSEHOLD_CONSUMPTION_EXPENDITURE_SURVEY_HCES_2023_2024/`

Item codes for 'Rice' include: 101, 102, and 061. See Section 5.1 of the above documentation for more context.
- Item code 101 signifies rice procured through PDS, using ration card.
- Item code 061 signifies rice procured through PDS, free of charge.
- Item code 102 signifies rice procured/consumed from other sources. (NOTE: Presumably this is unfortified rice.)

We also will need to approximate DHS wealth quintiles in the HCES data, by using similar variables as is used by the DHS to 
calculate wealth quintiles in India (see DHS wealth quintile documentation here: 
https://dhsprogram.com/programming/wealth%20index/India%20DHS%202015-16/India%202015-16%20sps.txt)

In this notebook, we extract the raw data from the HCES website and do some initial processing in order to create files 
that we can further tabulate based on our project needs in a later step. For now, we process these raw data into dataframes 
that have a row for each household/individual with the following columns:
- Whether or not rice was consumed (binary variable - this will be used to calculate coverage percentages later)
- Amount of PDS rice consumed (grams per day - currently in raw HCES, *I think* this is in kg per 2 weeks; this will be used to calculate fortified rice consumption amount) 
- Amount of non-PDS rice consumed (grams per day - currently in raw HCES, *I think* this is in kg per 2 weeks; this will be used to calculate unfortified rice consumption amount) 
- Wealth composite score/quintile 
- Sex/gender (will be used to calculate WBPRA)
- Age (will be used to tabulate age groups)



## Load data

In [ ]:
households, household_members_raw = hces_utils.load_households()
households

In [ ]:
# The fact that there are a couple very large households (e.g. 20+ members) is a little
# surprising - maybe we can investigate to confirm these are some kind of GQ.
households.hh_size.hist()

In [ ]:
# Bizarrely, I can't find documentation of these codes -- but we know the rent rate is only asked for rural
households.assign(rent_rate=households.rural_rent_rate_availability.notnull()).groupby(
    "sector"
).rent_rate.mean()

In [ ]:
households.sector.value_counts(dropna=False)

In [ ]:
households.isnull().mean().sort_values()

## Calculate wealth quintiles

In order to add wealth stratifications into our HCES data, we will emulate DHS methods for calculating wealth quintiles.
They don't ask something like "how wealthy are you?"
Instead, they ask "do you have a refrigerator?" and then a bunch of other questions like that,
and analyze them into a composite wealth index.
The way they do so is actually pretty weird and interesting, but I won't get into that here.
It is described at https://dhsprogram.com/topics/wealth-index/Wealth-Index-Construction.cfm.

In a first pass, we attempted to use the DHS weights directly.
See [this past version of the notebook](https://github.com/ihmeuw/vivarium_gates_lsff_2026_maternal/blob/12d63b4c5a3b7cb204c4f6eb3de7d6ffe87ce9c3/0100_data_prep/01_extract_hces.ipynb)
for how we started to do this.

However, this was tricky,
because none of the questions asked in HCES are exactly the same, with the same answer options,
as in DHS.
We could sort of use proxies and fudge things, but it was pretty rough.
Here, I have instead used the same *methods* used in the DHS to make an entirely new
model of the latent wealth variable.
I followed the steps described here: https://dhsprogram.com/programming/wealth%20index/Steps_to_constructing_the_new_DHS_Wealth_Index.pdf

However, even this is pretty underwhelming. Probably a better strategy to try in the future would be to make a more complex and
precise crosswalk between DHS and HCES questions/responses, then train a model in DHS to predict the DHS wealth quintile based
only on information available in HCES, and finally predict HCES.

DHS wealth index variables include: 
- Source of drinking water
- Type of toilet facility
- Electricity
- Mattress
- Pressure cooker
- Chair
- Cot or bed
- Table
- Electric fan
- Radio or transistor
- Black and white television
- Colour television
- Sewing machine
- Mobile telephone
- Telephone (non-mobile)
- Internet
- Computer
- Refrigerator
- Air conditioner/cooler
- Washing machine
- Watch or clock
- Bicycle
- Motorcycle or Scooter
- Animal-drawn cart
- Car
- Water pump
- Thresher
- Tractor
- Type of cooking fuel
- Main material of floor
- Main roof material
- Main wall material
- Hectares for agricultural land
- Out of this land, how much is irrigated?
- Cows / bulls / buffaloes
- Camels
- Horses / donkeys / mules
- Goats
- Sheep
- Chickens / ducks
- Bank account
- Members per sleeping room
- Compute urban and rural variables coded (1/0) for filters later
- Toilet facility by shared/not shared
- Land area by units - if there are separate units - need to convert them to one unit

HCES variables:
- Type of land owned
- What is the total area of all owned (owned and possessed or leased out) land (within the country) by the household as on the date of survey (area in acre)?(upto two places of decimal) 
- Primary source of energy of the household for cooking
- Any household member used the internet in the last 30 days (from the person-level level 2 records)
- Whether household possessed one or more item as on the date of the survey- Television 
- Whether household possessed one or more item as on the date of the survey- Radio
- Whether household possessed one or more item as on the date of the survey - Laptop/PC
- Whether household possessed one or more item as on the date of the survey- Mobile handset
- Whether household possessed one or more item as on the date of the survey- Bicycle
- Whether household possessed one or more item as on the date of the survey- Motorcycle, scooter 
- Whether household possessed one or more item as on the date of the survey- Motor car/jeep/van
- Whether household possessed one or more item as on the date of the survey- Trucks
- Whether household possessed one or more item as on the date of the survey - Animal cart
- Whether household possessed one or more item as on the date of the survey- Refrigerator
- Whether household possessed one or more item as on the date of the survey- Washing machine
- Whether household possessed one or more item as on the date of the survey- Air conditioner/air cooler 
- Type of multichannel television facility is used by the household as on the date of the survey

Crosswalk spreadsheet saved here: https://uwnetid.sharepoint.com/sites/ihme_simulation_science_team/_layouts/15/doc.aspx?sourcedoc={eddcbf51-3462-47d7-b847-79edea5046c6}&action=edit


In [ ]:
wealth_info, wealth_predictors = hces_utils.assign_wealth_quintiles(
    households, diagnostics=True
)
wealth_info

In [ ]:
wealth_info.combined_score.hist(bins=500)

In [ ]:
wealth_info.groupby(
    "wealth_quintile", observed=True
).multiplier.sum() / wealth_info.multiplier.sum()

In [ ]:
for column in wealth_predictors:
    print(column)
    print(
        wealth_info.groupby(
            ["wealth_quintile", column], observed=True
        ).multiplier.sum()
        / wealth_info.groupby("wealth_quintile").multiplier.sum()
    )

In [ ]:
# Whoa, owning land or a house is uncorrelated?!
india_dhs = pd.read_stata(
    "/snfs1/DATA/DHS_PROG_DHS/IND/2015_2016/IND_DHS7_2015_2016_HH_IAHR74FL_Y2018M12D06.DTA",
    columns=["hv005", "hv270", "sh46"],
).rename(columns={"hv005": "weight", "hv270": "wealth_quintile", "sh46": "owns_house"})

In [ ]:
india_dhs.groupby(
    ["wealth_quintile", "owns_house"], observed=True
).weight.sum() / india_dhs.groupby("wealth_quintile").weight.sum()

In [ ]:
# Bizarre. Even in DHS it's basically uncorrelated, although in HCES it seems to be reversed.

In [ ]:
households["wealth_quintile"] = wealth_info["wealth_quintile"]

In [ ]:
assert households.wealth_quintile.notnull().all()

## Household rice consumption

Item codes for 'Rice' include: 101, 102, and 061. See Section 5.1 of the above documentation for more context.

- Item code 101 signifies rice procured through PDS, using ration card.
- Item code 061 signifies rice procured through PDS, free of charge.
- Item code 102 signifies rice procured/consumed from other sources. (NOTE: Presumably this is unfortified rice.)

Units: kg/30 days.

In [ ]:
consumption = hces_utils.read_consumption([61, 101, 102])
consumption

In [ ]:
consumption["item_code"] = consumption.item_code.map(
    {
        61: "pds free rice",
        101: "pds paid rice",
        102: "other rice",
    }
)
consumption["is_pds"] = consumption.item_code.isin(["pds free rice", "pds paid rice"])

In [ ]:
consumption["source"] = consumption.source.map(
    {
        1: "only purchase",
        2: "only home-grown",
        3: "both purchase and home-grown",
        4: "only free collection",
        5: "only exchange",
        6: "only gifts",
        9: "other",
    }
)

In [ ]:
consumption["total_consumption_amt"] = consumption.total_consumption_amt.astype(float)
consumption["consumption_amt_home_produce"] = (
    consumption.consumption_amt_home_produce.astype(float)
)

In [ ]:
# I assume for these high-missingness columns, null means 0
consumption.isnull().mean()

In [ ]:
(consumption == 0).mean()

In [ ]:
consumption = consumption.fillna(0)

In [ ]:
consumption["pds_rice"] = np.where(
    consumption.is_pds,
    consumption.total_consumption_amt,
    0.0,
)

In [ ]:
consumption["non_pds_rice_purchased"] = np.where(
    consumption.is_pds,
    0.0,
    np.where(
        consumption.source.isin(["only purchase", "both purchase and home-grown"]),
        consumption.total_consumption_amt - consumption.consumption_amt_home_produce,
        0.0,
    ),
)

In [ ]:
consumption["non_pds_rice_non_purchased"] = (
    consumption.total_consumption_amt
    - consumption.pds_rice
    - consumption.non_pds_rice_purchased
)

In [ ]:
consumption = consumption.groupby("common_id")[
    ["pds_rice", "non_pds_rice_purchased", "non_pds_rice_non_purchased"]
].sum()
consumption

In [ ]:
consumption["rice"] = consumption.sum(axis=1)
consumption["proportion_non_pds_purchased"] = (
    consumption.non_pds_rice_purchased / (consumption.rice - consumption.pds_rice)
).clip(0, 1)
consumption

In [ ]:
len(consumption) / len(households)

In [ ]:
(households.merge(consumption, on="common_id", how="left").rice > 0).mean()

In [ ]:
consumption = consumption.reindex(households.common_id).fillna(0).reset_index()
consumption

In [ ]:
(households.merge(consumption, on="common_id", how="left").rice > 0).mean()

## Individual rice consumption

We distribute the rice to the individuals according to the number of meals they have eaten at home.
We assume that children eating school lunches are getting fortified rice, as the scale-up has also
happened through that program.
Otherwise, we assume that meals eaten away from the home use rice and fortified rice at the same rate
as meals at home.

In [ ]:
# Level 2 was read once with the households (for the internet indicator); reuse it.
household_members = hces_utils.load_household_members(household_members_raw, households)
household_members

In [ ]:
household_members = hces_utils.add_total_meals(household_members)

In [ ]:
household_members.plot(kind="scatter", x="usual_daily_meals", y="meals", alpha=0.01)

In [ ]:
household_meals = household_members.groupby(
    "common_id"
).meals_at_home_last_30_days.sum()
household_meals

In [ ]:
consumption = consumption.merge(
    household_meals, on="common_id", validate="1:1", how="left"
)
consumption

In [ ]:
consumption.isnull().mean()

In [ ]:
consumption = consumption.dropna(how="any").copy()
consumption

In [ ]:
consumption["rice_per_meal_kg"] = np.where(
    consumption.rice == 0,
    0.0,
    consumption.rice / consumption.meals_at_home_last_30_days,
)

In [ ]:
# In a small number of cases, rice consumption is reported despite very few or no meals at home
impossible_rice_consumption = (consumption.rice > 0) & (
    consumption.meals_at_home_last_30_days == 0
)
consumption[impossible_rice_consumption]

In [ ]:
# This must be wrong somehow, so we clip it
consumption["rice_per_meal_kg"] = consumption["rice_per_meal_kg"].clip(
    upper=np.percentile(consumption.rice_per_meal_kg, 95)
)

In [ ]:
consumption[consumption.rice_per_meal_kg.notnull()].sort_values("rice_per_meal_kg")
consumption.rice_per_meal_kg.hist()

In [ ]:
consumption["pds_rice_per_meal_kg"] = consumption.rice_per_meal_kg * (
    consumption.pds_rice / consumption.rice
)
consumption.pds_rice_per_meal_kg.hist()

In [ ]:
household_members = household_members.merge(
    consumption.reset_index()[
        [
            "common_id",
            "rice_per_meal_kg",
            "pds_rice_per_meal_kg",
            "proportion_non_pds_purchased",
        ]
    ],
    on="common_id",
    how="left",
    validate="m:1",
)
household_members

In [ ]:
household_members.isnull().mean()

In [ ]:
household_members["government_meals_away_from_home"] = (
    household_members.meals_from_school_balwadi_last_30_days
)
household_members["non_government_meals_away_from_home"] = (
    household_members.meals
    - household_members.meals_at_home_last_30_days
    - household_members.government_meals_away_from_home
)

non_government_meals_notnull = (
    household_members.non_government_meals_away_from_home.notnull()
)
assert (
    household_members.loc[
        non_government_meals_notnull, "non_government_meals_away_from_home"
    ]
    >= 0
).all()


In [ ]:
household_members["rice_at_home"] = (
    household_members.meals_at_home_last_30_days * household_members.rice_per_meal_kg
)
household_members["rice_away_from_home_in_government_meals"] = (
    household_members.government_meals_away_from_home
    * household_members.rice_per_meal_kg
)
household_members["rice_away_from_home_in_non_government_meals"] = (
    household_members.non_government_meals_away_from_home
    * household_members.rice_per_meal_kg
)
household_members["rice"] = household_members.meals * household_members.rice_per_meal_kg
assert np.allclose(household_members.rice_at_home + household_members.rice_away_from_home_in_government_meals + household_members.rice_away_from_home_in_non_government_meals, household_members.rice, rtol=0, atol=1e-5)

In [ ]:
household_members["government_rice_at_home"] = (
    household_members.meals_at_home_last_30_days
    * household_members.pds_rice_per_meal_kg
)
assert (household_members[
    household_members.government_rice_at_home.notnull()
].government_rice_at_home <= household_members[
    household_members.government_rice_at_home.notnull()
].rice_at_home).all()

household_members["government_rice_away_from_home_in_government_meals"] = (
    household_members.government_meals_away_from_home
    * household_members.rice_per_meal_kg
)
household_members["government_rice_away_from_home_in_non_government_meals"] = (
    household_members.non_government_meals_away_from_home
    # NOTE: Assume same as at-home meals
    * household_members.pds_rice_per_meal_kg
)
household_members["government_rice"] = (
    household_members.government_rice_at_home
    + household_members.government_rice_away_from_home_in_government_meals
    + household_members.government_rice_away_from_home_in_non_government_meals
)
government_rice_notnull = household_members.government_rice.notnull()

In [ ]:
# Except for floating-point error, rice >= government_rice
assert (household_members[government_rice_notnull].government_rice <= household_members[government_rice_notnull].rice + 1e-6).all()

In [ ]:
household_members.loc[government_rice_notnull, "government_rice"] = household_members.loc[government_rice_notnull, "government_rice"].clip(upper=household_members.loc[government_rice_notnull, "rice"])

In [ ]:
household_members["non_government_rice"] = (household_members.rice - household_members.government_rice)
household_members["non_government_rice_at_home"] = (household_members.rice_at_home - household_members.government_rice_at_home)
household_members["non_government_rice_away_from_home_in_non_government_meals"] = (
    household_members.rice_away_from_home_in_non_government_meals -
    household_members.government_rice_away_from_home_in_non_government_meals
)

In [ ]:
assert (~government_rice_notnull | np.isclose(
    household_members.non_government_rice_at_home +
    household_members.non_government_rice_away_from_home_in_non_government_meals,
    household_members.non_government_rice,
    rtol=0,
    atol=1e-6,
)).all()

In [ ]:
household_members["proportion_government"] = (
    household_members.government_rice / household_members.rice
)
household_members["proportion_non_government_rice_purchased"] = (
    (
        (
            household_members.proportion_non_pds_purchased
            * household_members.non_government_rice_at_home
        ) + (
            # NOTE: We assume that the purchased proportion is the same in non-government meals away from home
            # Another reasonable assumption would be that all non-government rice in non-government meals away from home is purchased;
            # but this is more conservative with respect to the effect of fortification.
            household_members.proportion_non_pds_purchased
            * household_members.non_government_rice_away_from_home_in_non_government_meals
        )
    ) /
    household_members.non_government_rice
)

In [ ]:
# Within floating point error of respecting bounds
purchased_notnull = household_members.proportion_non_government_rice_purchased.notnull()
assert (
    household_members.loc[purchased_notnull, "proportion_non_government_rice_purchased"]
    .between(-1e-6, 1 + 1e-6)
    .all()
)
household_members.loc[purchased_notnull, "proportion_non_government_rice_purchased"] = (
    household_members.loc[purchased_notnull, "proportion_non_government_rice_purchased"]
    .clip(lower=0, upper=1)
)

In [ ]:
household_members

In [ ]:
household_members.isnull().mean()

In [ ]:
# Convert kg/30 days to g/day
household_members["rice_g_per_day"] = household_members.rice / 30 * 1_000

In [ ]:
def save_data(series, name):
    hces_utils.save_data(series, name, vehicle="rice")


def save_for_each_fortificant(series, name):
    save_data(series, f"iron/{name}")
    save_data(series, f"folate/{name}")

In [ ]:
weighted_avg_and_std(
    household_members.rice_g_per_day > 0, weights=household_members.multiplier
)

In [ ]:
weighted_summary(
    household_members, lambda df: df.rice_g_per_day > 0, by="wealth_quintile"
)

In [ ]:
# NOTE: We do not have pregnancy status in this survey, so we assume it has no effect on consumption!
any_consumption = weighted_summary(
    household_members,
    lambda df: np.where(df.rice_g_per_day.notnull(), df.rice_g_per_day > 0, np.nan),
)
any_consumption

In [ ]:
save_data(any_consumption["mean"], "rice/vehicle_consumption/any")

In [ ]:
weighted_avg_and_std(
    household_members.rice_g_per_day, weights=household_members.multiplier
)

In [ ]:
weighted_summary(household_members, lambda df: df.rice_g_per_day, by="wealth_quintile")

In [ ]:
consumption_amount = weighted_summary(household_members, lambda df: df.rice_g_per_day)
consumption_amount

In [ ]:
save_data(consumption_amount["mean"], "rice/vehicle_consumption/amount/mean")
save_data(consumption_amount["sd"], "rice/vehicle_consumption/amount/sd")

In [ ]:
# Baseline fortification of government rice (PDS, plus school and anganwadi meals).
# This notebook writes the shares of people eating *any* and *only* government rice. The share of
# government rice that is actually fortified is NOT applied here: data prep (extraction/prep_extracted.ipynb)
# multiplies these shares by the India rice "Vehicle fortification at baseline -- any" value from
# the extraction tables. (It used to be hard-coded here as GOVERNMENT_BASELINE_COVERAGE = 0.8.)
# We apply maximum individual heterogeneity, meaning that the people who are missed
# are *completely* missed.

In [ ]:
weighted_avg_and_std(
    household_members.proportion_government,
    weights=household_members.multiplier,
)

In [ ]:
weighted_summary(
    household_members, lambda df: df.proportion_government, by="wealth_quintile"
)

In [ ]:
any_government_rice = weighted_summary(
    household_members,
    lambda df: np.where(
        df.proportion_government.notnull(),
        (df.proportion_government > 0).astype(float),
        np.nan,
    ),
)
any_government_rice

In [ ]:
# Save locally; prep_extracted scales it by the share of government rice fortified
any_government_rice["mean"].rename("value").to_csv(
    "india_share_eating_any_government_rice.csv"
)

In [ ]:
only_government_rice = weighted_summary(
    household_members,
    lambda df: np.where(
        df.proportion_government.notnull(),
        (df.proportion_government == 1).astype(float),
        np.nan,
    ),
)
only_government_rice

In [ ]:
# Save locally; prep_extracted scales it by the share of government rice fortified
only_government_rice["mean"].rename("value").to_csv(
    "india_share_eating_only_government_rice.csv"
)

In [ ]:
# Remember, we've assumed maximum heterogeneity here, so the people who are partially covered
# have *all* of their government rice fortified.
partial_fortification_amount = weighted_summary(
    household_members[
        (household_members.proportion_government > 0)
        & (household_members.proportion_government < 1)
    ],
    lambda df: df.proportion_government,
)
partial_fortification_amount

In [ ]:
save_for_each_fortificant(
    partial_fortification_amount["mean"],
    "rice/baseline_fortification/partial_coverage_amount/mean",
)
save_for_each_fortificant(
    partial_fortification_amount["sd"],
    "rice/baseline_fortification/partial_coverage_amount/sd",
)

In [ ]:
weighted_avg_and_std(
    household_members.proportion_government, weights=household_members.multiplier
)

In [ ]:
proportion_government = weighted_summary(
    household_members,
    lambda df: df.proportion_government,
    # Rice-quantity weights: prep_extracted composes this share with the
    # rice-weighted purchased proportion, so both must use the volume frame.
    weights=lambda df: df.multiplier * df.rice,
)
proportion_government

In [ ]:
# Save locally for further scaling in the extraction notebook
proportion_government["mean"].rename("value").to_csv(
    "india_proportion_government_rice.csv"
)

In [ ]:
weighted_summary(
    household_members,
    lambda df: df.proportion_non_government_rice_purchased,
    weights=lambda df: df.multiplier * df.non_government_rice,
    by="wealth_quintile",
)

In [ ]:
proportion_non_government_rice_purchased = weighted_summary(
    household_members,
    lambda df: df.proportion_non_government_rice_purchased,
    weights=lambda df: df.multiplier * df.non_government_rice,
)
proportion_non_government_rice_purchased

In [ ]:
proportion_non_government_rice_purchased["mean"].rename("value").to_csv(
    "india_non_government_rice_proportion_purchased.csv"
)